# Limpieza de columnas numericas en `facturacion_ventas.csv`

Este notebook limpia columnas numericas de `tablas/facturacion_ventas.csv` para que valores como `54000.0` queden como `54000`.

- Lee el CSV con separador `;`.
- Limpia solo las columnas definidas en `NUMERIC_COLUMNS`.
- Mantiene celdas vacias como vacias.
- Crea un backup antes de sobrescribir el archivo original.
- Puede repetirse cuando se carguen mas datos.

Nota: `fecha de pago 2` no se limpia como numero porque es una columna de fecha.

In [ ]:
from datetime import datetime
from pathlib import Path
import re
import shutil

import pandas as pd

In [ ]:
# Si ejecutas este notebook desde la carpeta del repo, PROJECT_ROOT queda en el repo.
# Si lo ejecutas desde la carpeta "scripts de limpieza", PROJECT_ROOT sube un nivel.
CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "scripts de limpieza" else CURRENT_DIR

CSV_PATH = PROJECT_ROOT / "tablas" / "facturacion_ventas.csv"
BACKUP_DIR = PROJECT_ROOT / "tablas" / "backups"

NUMERIC_COLUMNS = [
    "pago efectivo",
    "pago tarjeta",
    "pago transferencia",
    "total pagado 1",
    "valor factura",
    "pendiente",
    "pago efectivo 2",
    "pago tarjeta 2",
    "pago transferencia 2",
]

CSV_PATH

In [ ]:
df = pd.read_csv(
    CSV_PATH,
    sep=";",
    dtype=str,
    keep_default_na=False,
)

print(f"Filas: {len(df):,}")
print(f"Columnas: {len(df.columns):,}")

missing_columns = [column for column in NUMERIC_COLUMNS if column not in df.columns]
if missing_columns:
    print("Columnas no encontradas:")
    for column in missing_columns:
        print(f"- {column}")
else:
    print("Todas las columnas numericas fueron encontradas.")

In [ ]:
def clean_number_text(value):
    """Convierte textos numericos como '54000.0' en '54000'.

    Mantiene vacios como vacios. Si aparece un decimal no cero, se trunca la
    parte decimal porque el objetivo de estas columnas es guardar pesos enteros.
    """
    text = str(value).strip()

    if text == "":
        return ""

    # Quita simbolos comunes sin tocar el separador decimal.
    text = (
        text.replace("$", "")
        .replace(" ", "")
        .replace(",", "")
    )

    # Casos tipo 54000.0, 54000.00 o 54000.75 -> 54000.
    if re.fullmatch(r"-?\d+\.\d+", text):
        return text.split(".", 1)[0]

    # Casos tipo 54000.
    if re.fullmatch(r"-?\d+\.", text):
        return text[:-1]

    return text

In [ ]:
existing_numeric_columns = [
    column for column in NUMERIC_COLUMNS
    if column in df.columns
]

before_preview = df[existing_numeric_columns].tail(10).copy()

changes_by_column = {}
for column in existing_numeric_columns:
    before = df[column].copy()
    df[column] = df[column].map(clean_number_text)
    changes_by_column[column] = int((before != df[column]).sum())

after_preview = df[existing_numeric_columns].tail(10).copy()

pd.DataFrame({
    "columna": list(changes_by_column.keys()),
    "celdas modificadas": list(changes_by_column.values()),
})

## Vista previa antes de guardar

In [ ]:
before_preview

## Vista previa despues de limpiar

In [ ]:
after_preview

In [ ]:
# Cambia a False si NO quieres crear backup.
CREATE_BACKUP = True

if CREATE_BACKUP:
    BACKUP_DIR.mkdir(parents=True, exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    backup_path = BACKUP_DIR / f"facturacion_ventas_backup_{timestamp}.csv"
    shutil.copy2(CSV_PATH, backup_path)
    print(f"Backup creado: {backup_path}")

df.to_csv(
    CSV_PATH,
    sep=";",
    index=False,
    encoding="utf-8",
)

print(f"Archivo limpio guardado: {CSV_PATH}")